# TennisMyLife — Gallica 1903 Colab worker

Run the cells from top to bottom. When asked for a file, select **`tml_colab_ed25519`** from your Windows Downloads folder.

This notebook reads the active 1903 Colab claim from the VPS, executes the assigned ALTO and RapidOCR branches, and uploads each completed page immediately back to the VPS.


In [ ]:
!rm -rf /content/Tennis-OCR-Pipeline
!git clone -q https://github.com/Tennismylife/Tennis-OCR-Pipeline.git /content/Tennis-OCR-Pipeline
!pip -q install -r /content/Tennis-OCR-Pipeline/colab/requirements.txt


In [ ]:
from google.colab import files
import base64
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No SSH key uploaded')
key_name, key_bytes = next(iter(uploaded.items()))
if key_name.endswith('.pub'):
    raise RuntimeError('Upload the private key tml_colab_ed25519, not the .pub file')
KEY_B64 = base64.b64encode(key_bytes).decode()
print('SSH key loaded:', key_name)


In [ ]:
import sys
sys.path.insert(0, '/content/Tennis-OCR-Pipeline/colab')
from worker import connect_sftp

VPS_HOST = 'vibrant-lovelace.82-165-11-122.plesk.page'
VPS_USER = 'andre'
VPS_PORT = 22
VPS_MANIFEST = '/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/00_MANIFEST/colab_active_claims.tsv'
VPS_REMOTE_CACHE = '/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/ocr_cache_latin_full/targeted_remaining_1903'
VPS_ALTO_CACHE = '/home/andre/GallicaJobs/_shared/alto_cache'

tr, sftp = connect_sftp(VPS_HOST, VPS_USER, KEY_B64, VPS_PORT)
sftp.get(VPS_MANIFEST, '/content/colab_claim.tsv')
sftp.close(); tr.close()
rows = sum(1 for _ in open('/content/colab_claim.tsv', encoding='utf-8-sig')) - 1
print('Claim downloaded. Rows:', rows)


In [ ]:
import subprocess
cmd = [
    'python', '/content/Tennis-OCR-Pipeline/colab/worker.py',
    '--manifest', '/content/colab_claim.tsv',
    '--vps-host', VPS_HOST, '--vps-user', VPS_USER, '--vps-key-b64', KEY_B64,
    '--vps-port', str(VPS_PORT),
    '--remote-cache', VPS_REMOTE_CACHE,
    '--remote-alto-cache', VPS_ALTO_CACHE,
    '--profile', 'HQ',
    '--delay', '15',
    '--max-pages', '0'
]
subprocess.run(cmd, check=True)


If the Colab runtime stops, reopen this notebook and run the cells again. Pages already uploaded to the VPS are detected and skipped.
